# Incident Response Runbook: Execution - Command and Scripting Interpreter

**Tactic:** Execution
**Technique:** Command and Scripting Interpreter (T1059)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Command and Scripting Interpreter activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add project root to path for imports
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
suspicious_activities = []
incident_id = None

# Query Splunk for command and scripting interpreter indicators
print(f"\n[DETECTION] Querying Splunk for command and scripting interpreter activities...")
try:
    # PowerShell suspicious execution
    powershell_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4688 "powershell.exe" OR "pwsh.exe"
    | eval risk_score = case(
        match(CommandLine, "bypass|encodedcommand|invoke-expression"), 10,
        match(CommandLine, "downloadstring|downloadfile|net.webclient"), 9,
        match(CommandLine, "iex|invoke-command|scriptblock"), 8,
        match(CommandLine, "-nop|-noprofile|-executionpolicy bypass"), 7,
        match(CommandLine, "base64|compressed"), 6,
        1==1, 3
    )
    | where risk_score >= 6
    | stats count by host, user, CommandLine, risk_score
    | sort -risk_score
    """

    powershell_results = splunk.search(powershell_query, timeframe="-24h")
    if powershell_results:
        print(f"   Found {len(powershell_results)} suspicious PowerShell executions")
        for result in powershell_results[:10]:  # Top 10
            suspicious_activities.append({
                'type': 'powershell_execution',
                'host': result.get('host'),
                'user': result.get('user'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

    # Command prompt suspicious execution
    cmd_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4688 "cmd.exe"
    | eval risk_score = case(
        match(CommandLine, "/c.*echo|for.*do|findstr.*suspicious"), 8,
        match(CommandLine, "net user|net group|whoami /all"), 7,
        match(CommandLine, "schtasks|at.exe|bitsadmin"), 9,
        match(CommandLine, "reg add|reg delete|sc create"), 8,
        1==1, 4
    )
    | where risk_score >= 6
    | stats count by host, user, CommandLine, risk_score
    | sort -risk_score
    """

    cmd_results = splunk.search(cmd_query, timeframe="-24h")
    if cmd_results:
        print(f"   Found {len(cmd_results)} suspicious command prompt executions")
        for result in cmd_results[:10]:  # Top 10
            suspicious_activities.append({
                'type': 'cmd_execution',
                'host': result.get('host'),
                'user': result.get('user'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

    # Python suspicious execution
    python_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4688 "python.exe" OR "python3.exe"
    | eval risk_score = case(
        match(CommandLine, "exec|eval|subprocess|socket|requests"), 8,
        match(CommandLine, "base64|decode|encode"), 7,
        match(CommandLine, "-c.*import"), 6,
        match(CommandLine, "ctypes|winreg|_winapi"), 9,
        1==1, 3
    )
    | where risk_score >= 6
    | stats count by host, user, CommandLine, risk_score
    | sort -risk_score
    """

    python_results = splunk.search(python_query, timeframe="-24h")
    if python_results:
        print(f"   Found {len(python_results)} suspicious Python executions")
        for result in python_results[:10]:  # Top 10
            suspicious_activities.append({
                'type': 'python_execution',
                'host': result.get('host'),
                'user': result.get('user'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

except Exception as e:
    print(f"   Splunk query failed: {e}")

# Get unique affected systems
unique_hosts = set()
for activity in suspicious_activities:
    if activity.get('host'):
        unique_hosts.add(activity['host'])

# Check CrowdStrike for endpoint details
print(f"\n[DETECTION] Checking CrowdStrike for endpoint details...")
for host in unique_hosts:
    try:
        cs_info = crowdstrike.get_host_info(host)
        if cs_info:
            affected_systems.append({
                'hostname': host,
                'device_id': cs_info.get('device_id'),
                'os': cs_info.get('os_version'),
                'last_seen': cs_info.get('last_seen'),
                'activities': [a for a in suspicious_activities if a.get('host') == host]
            })
            print(f"   Found endpoint details for: {host}")
    except Exception as e:
        print(f"   CrowdStrike check failed for {host}: {e}")

# Enrich with threat intelligence
print(f"\n[INTEL] Enriching with threat intelligence...")
for activity in suspicious_activities[:5]:  # Check top 5
    try:
        if activity.get('command'):
            # Extract suspicious patterns from commands
            suspicious_patterns = re.findall(r'\b(bypass|encodedcommand|downloadstring|iex|net\.webclient|subprocess|socket)\b', activity['command'], re.IGNORECASE)
            for pattern in suspicious_patterns:
                ti_data = misp.lookup_command_pattern(pattern)
                if ti_data:
                    activity['threat_intel'] = ti_data
                    print(f"   Threat intel found for command pattern: {pattern}")
                    break
    except Exception as e:
        print(f"   Threat intelligence lookup failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    case_data = {
        'title': f'Command and Scripting Interpreter Activities - {len(suspicious_activities)} indicators',
        'description': f'Detected suspicious command and scripting interpreter activities across {len(affected_systems)} systems',
        'severity': 'HIGH',
        'tactic': 'Execution',
        'technique': 'Command and Scripting Interpreter (T1059)',
        'indicators': suspicious_activities[:10],  # Top 10 indicators
        'detection_time': detection_time
    }
    incident_id = iris.create_case(case_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious scripting activities detected")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - Threat intelligence enriched for {len([a for a in suspicious_activities if a.get('threat_intel')])} activities")
print(f"  - IRIS case created: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_hosts = []
blocked_entities = []
disabled_interpreters = []

# Isolate affected systems immediately
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'isolated':
            isolated_hosts.append(system['hostname'])
            print(f"   Isolated {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result.get('message', 'unknown error')}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Block associated IPs and domains
print(f"\n[ACTION] Blocking associated IPs and domains...")
for activity in suspicious_activities:
    try:
        if activity.get('src_ip'):
            block_result = shuffle.block_ip(activity['src_ip'])
            if block_result.get('status') == 'blocked':
                blocked_entities.append(f"IP:{activity['src_ip']}")
                print(f"   Blocked source IP: {activity['src_ip']}")

        if activity.get('domain'):
            domain_block = shuffle.block_domain(activity['domain'])
            if domain_block.get('status') == 'blocked':
                blocked_entities.append(f"Domain:{activity['domain']}")
                print(f"   Blocked domain: {activity['domain']}")
    except Exception as e:
        print(f"   Entity blocking failed: {e}")

# Disable scripting interpreters temporarily
print(f"\n[ACTION] Disabling scripting interpreters temporarily...")
for system in affected_systems:
    try:
        interpreters = ['powershell.exe', 'cmd.exe', 'python.exe', 'bash', 'sh']
        for interpreter in interpreters:
            disable_result = crowdstrike.disable_interpreter(system['device_id'], interpreter)
            if disable_result.get('status') == 'disabled':
                disabled_interpreters.append(f"{system['hostname']}:{interpreter}")
                print(f"   Disabled {interpreter} on {system['hostname']}")
    except Exception as e:
        print(f"   Interpreter disabling failed for {system.get('hostname', 'unknown')}: {e}")

# Terminate suspicious scripting processes
print(f"\n[ACTION] Terminating suspicious scripting processes...")
terminated_processes = []
for activity in suspicious_activities:
    try:
        if activity.get('process_id') and activity.get('host'):
            termination_result = crowdstrike.terminate_process(
                activity['host'],
                activity['process_id']
            )
            if termination_result.get('status') == 'terminated':
                terminated_processes.append(f"{activity['host']}:{activity['process_id']}")
                print(f"   Terminated scripting process {activity['process_id']} on {activity['host']}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Quarantine executed scripts
print(f"\n[ACTION] Quarantining executed scripts...")
quarantined_scripts = []
for system in affected_systems:
    try:
        for script in system.get('executed_scripts', []):
            quarantine_result = crowdstrike.quarantine_file(
                system['hostname'],
                script.get('path', '')
            )
            if quarantine_result.get('status') == 'quarantined':
                quarantined_scripts.append(script.get('path', ''))
                print(f"   Quarantined script: {script.get('path', '')}")
    except Exception as e:
        print(f"   Script quarantine failed: {e}")

# Enable enhanced monitoring for scripting execution
print(f"\n[ACTION] Enabling enhanced monitoring for scripting execution...")
try:
    monitoring_result = shuffle.enable_enhanced_monitoring(
        targets=[s['hostname'] for s in affected_systems],
        rules=['scripting_execution_detection', 'interpreter_anomaly', 'suspicious_command_patterns']
    )
    if monitoring_result.get('status') == 'enabled':
        print(f"   Enhanced scripting monitoring enabled for {len(affected_systems)} systems")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} systems isolated")
print(f"  - {len(blocked_entities)} entities blocked")
print(f"  - {len(disabled_interpreters)} interpreters disabled")
print(f"  - {len(terminated_processes)} suspicious processes terminated")
print(f"  - {len(quarantined_scripts)} scripts quarantined")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_scripts = []
cleaned_artifacts = []
hardened_systems = []

# Remove executed scripts and artifacts
print(f"\n[ACTION] Removing executed scripts and artifacts...")
for system in affected_systems:
    try:
        for script in system.get('executed_scripts', []):
            removal_result = crowdstrike.remove_file(
                system['hostname'],
                script.get('path', '')
            )
            if removal_result.get('status') == 'removed':
                removed_scripts.append(script.get('path', ''))
                print(f"   Removed executed script: {script.get('path', '')}")
    except Exception as e:
        print(f"   Script removal failed: {e}")

# Clean command history and caches
print(f"\n[ACTION] Cleaning command history and caches...")
for system in affected_systems:
    try:
        cleanup_result = crowdstrike.clean_command_history(system['device_id'])
        if cleanup_result.get('status') == 'cleaned':
            cleaned_artifacts.append(system['hostname'])
            print(f"   Cleaned command history and caches on {system['hostname']}")
    except Exception as e:
        print(f"   History cleanup failed for {system.get('hostname', 'unknown')}: {e}")

# Remove persistence mechanisms from scripts
print(f"\n[ACTION] Removing persistence mechanisms from scripts...")
removed_persistence = []
for system in affected_systems:
    try:
        persistence_result = crowdstrike.remove_script_persistence(system['device_id'])
        if persistence_result.get('status') == 'removed':
            removed_persistence.append(system['hostname'])
            print(f"   Removed script persistence mechanisms from {system['hostname']}")
    except Exception as e:
        print(f"   Persistence removal failed for {system.get('hostname', 'unknown')}: {e}")

# Harden scripting environments
print(f"\n[ACTION] Hardening scripting environments...")
for system in affected_systems:
    try:
        hardening_result = crowdstrike.harden_scripting_environment(system['device_id'])
        if hardening_result.get('status') == 'hardened':
            hardened_systems.append(system['hostname'])
            print(f"   Hardened scripting environment on {system['hostname']}")
    except Exception as e:
        print(f"   Environment hardening failed for {system.get('hostname', 'unknown')}: {e}")

# Implement script execution restrictions
print(f"\n[ACTION] Implementing script execution restrictions...")
restricted_systems = []
try:
    restriction_result = shuffle.implement_script_restrictions(
        targets=[s['hostname'] for s in affected_systems],
        policies={
            'unsigned_script_blocking': True,
            'remote_script_execution_disabled': True,
            'powershell_constrained_language': True
        }
    )
    if restriction_result.get('status') == 'implemented':
        restricted_systems.extend([s['hostname'] for s in affected_systems])
        print(f"   Implemented script execution restrictions on {len(affected_systems)} systems")
except Exception as e:
    print(f"   Script restriction implementation failed: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_scripting_threat_removal(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('scripting_threats_detected', True) == False else 'threats_remaining',
            'remaining_indicators': verify_result.get('remaining_indicators', 0)
        })
        status = "" if verify_result.get('scripting_threats_detected', True) == False else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('remaining_indicators', 0)} scripting indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_scripts)} executed scripts removed")
print(f"  - {len(cleaned_artifacts)} systems had command history cleaned")
print(f"  - {len(removed_persistence)} systems had persistence mechanisms removed")
print(f"  - {len(hardened_systems)} systems had scripting environments hardened")
print(f"  - {len(restricted_systems)} systems had script execution restrictions implemented")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
reconnected_hosts = []
validated_scripting = []

# Restore systems from clean backups
print(f"\n[ACTION] Restoring systems from clean backups...")
for system in affected_systems:
    try:
        if system['hostname'] in isolated_hosts:
            restore_result = crowdstrike.restore_from_backup(
                system['device_id'],
                backup_type='clean'
            )
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored {system['hostname']} from clean backup")
            else:
                print(f"   No clean backup available for {system['hostname']}, manual restoration required")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Reconnect isolated systems
print(f"\n[ACTION] Reconnecting isolated systems...")
for system in affected_systems:
    try:
        if system['hostname'] in isolated_hosts:
            reconnect_result = crowdstrike.reconnect_host(system['device_id'])
            if reconnect_result.get('status') == 'reconnected':
                reconnected_hosts.append(system['hostname'])
                print(f"   Reconnected {system['hostname']} to network")
    except Exception as e:
        print(f"   Reconnection failed for {system.get('hostname', 'unknown')}: {e}")

# Restore legitimate scripting capabilities
print(f"\n[ACTION] Restoring legitimate scripting capabilities...")
for system in affected_systems:
    try:
        if system['hostname'] in isolated_hosts:
            restore_result = crowdstrike.restore_scripting_capabilities(system['device_id'])
            if restore_result.get('status') == 'restored':
                validated_scripting.append(system['hostname'])
                print(f"   Restored legitimate scripting capabilities on {system['hostname']}")
    except Exception as e:
        print(f"   Scripting capability restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Validate scripting environment integrity
print(f"\n[ACTION] Validating scripting environment integrity...")
integrity_checks = []
for system in affected_systems:
    try:
        integrity_result = crowdstrike.validate_scripting_integrity(system['device_id'])
        integrity_checks.append({
            'system': system['hostname'],
            'integrity': integrity_result.get('integrity_score', 0),
            'issues': integrity_result.get('issues_found', [])
        })
        status = "" if integrity_result.get('integrity_score', 0) > 80 else ""
        print(f"  {status} Scripting integrity for {system['hostname']}: {integrity_result.get('integrity_score', 0)}% ({len(integrity_result.get('issues_found', []))} issues)")
    except Exception as e:
        print(f"   Integrity validation failed for {system.get('hostname', 'unknown')}: {e}")

# Implement application allowlisting
print(f"\n[ACTION] Implementing application allowlisting...")
allowlist_systems = []
try:
    allowlist_result = shuffle.implement_application_allowlisting(
        targets=[s['hostname'] for s in affected_systems],
        allowed_applications=['powershell.exe', 'cmd.exe', 'python.exe', 'bash', 'sh'],
        enforcement_mode='audit'
    )
    if allowlist_result.get('status') == 'implemented':
        allowlist_systems.extend([s['hostname'] for s in affected_systems])
        print(f"   Implemented application allowlisting on {len(affected_systems)} systems")
except Exception as e:
    print(f"   Application allowlisting failed: {e}")

# Set up continuous scripting monitoring
print(f"\n[ACTION] Setting up continuous scripting monitoring...")
monitoring_systems = []
for system in affected_systems:
    try:
        scripting_monitoring = crowdstrike.setup_scripting_monitoring(
            system['device_id'],
            rules=['anomalous_script_execution', 'suspicious_command_patterns', 'interpreter_abuse']
        )
        if scripting_monitoring.get('status') == 'monitoring':
            monitoring_systems.append(system['hostname'])
            print(f"   Continuous scripting monitoring enabled for {system['hostname']}")
    except Exception as e:
        print(f"   Scripting monitoring setup failed for {system.get('hostname', 'unknown')}: {e}")

# Update security policies
print(f"\n[ACTION] Updating security policies to prevent scripting attacks...")
policy_updates = []
try:
    # Enhance script execution monitoring
    script_policy = crowdstrike.update_security_policy(
        policy_type='script_execution_monitoring',
        rules={
            'script_blocking_enabled': True,
            'amsi_integration': True,
            'script_logging': True,
            'suspicious_pattern_detection': True
        }
    )
    if script_policy.get('status') == 'updated':
        policy_updates.append('script_execution_monitoring')
        print(f"   Updated script execution monitoring policy")

    # Update behavioral analytics
    behavior_policy = crowdstrike.update_security_policy(
        policy_type='behavioral_analytics',
        rules={
            'script_anomaly_detection': True,
            'command_pattern_analysis': True,
            'interpreter_behavior_monitoring': True
        }
    )
    if behavior_policy.get('status') == 'updated':
        policy_updates.append('behavioral_analytics')
        print(f"   Updated behavioral analytics policy")
except Exception as e:
    print(f"   Policy update failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from backup")
print(f"  - {len(reconnected_hosts)} systems reconnected")
print(f"  - {len(validated_scripting)} systems had scripting capabilities restored")
print(f"  - {len(allowlist_systems)} systems had application allowlisting implemented")
print(f"  - {len(monitoring_systems)} systems under continuous scripting monitoring")
print(f"  - {len(policy_updates)} security policies updated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Document incident details
incident_summary = {
    'incident_id': alert_data.get('incident_id', 'unknown'),
    'technique': 'Execution - Command and Scripting Interpreter',
    'detection_time': alert_data.get('timestamp', 'unknown'),
    'affected_systems': len(affected_systems),
    'scripting_indicators': len(suspicious_activities),
    'executed_scripts': sum(len(s.get('executed_scripts', [])) for s in affected_systems),
    'containment_actions': len(isolated_hosts) + len(blocked_entities) + len(disabled_interpreters),
    'eradication_actions': len(removed_scripts) + len(cleaned_artifacts) + len(hardened_systems),
    'recovery_actions': len(restored_systems) + len(reconnected_hosts) + len(validated_scripting),
    'timeline': {
        'detection': alert_data.get('timestamp', 'unknown'),
        'containment': 'completed',
        'eradication': 'completed',
        'recovery': 'completed',
        'closure': 'now'
    }
}

# Create comprehensive incident report
print(f"\n[ACTION] Creating comprehensive incident report...")
try:
    iris_report = iris.create_incident_report(
        incident_id=alert_data.get('incident_id', 'unknown'),
        title=f"Execution - Command and Scripting Interpreter Incident {alert_data.get('incident_id', 'unknown')}",
        description=f"Detected and responded to command and scripting interpreter activities on {len(affected_systems)} systems. {len(suspicious_activities)} suspicious indicators were identified and {sum(len(s.get('executed_scripts', [])) for s in affected_systems)} scripts were executed.",
        severity='high',
        status='closed',
        details={
            'technique': 'T1059 - Command and Scripting Interpreter',
            'indicators': [activity.get('description', 'Unknown scripting activity') for activity in suspicious_activities[:10]],
            'affected_assets': [s['hostname'] for s in affected_systems],
            'executed_scripts': [script.get('path', 'Unknown') for s in affected_systems for script in s.get('executed_scripts', [])],
            'response_actions': {
                'detection': f"Splunk queries identified {len(suspicious_activities)} scripting execution indicators",
                'containment': f"Isolated {len(isolated_hosts)} hosts, blocked {len(blocked_entities)} entities, disabled {len(disabled_interpreters)} interpreters",
                'eradication': f"Removed {len(removed_scripts)} scripts, cleaned {len(cleaned_artifacts)} artifacts, hardened {len(hardened_systems)} systems",
                'recovery': f"Restored {len(restored_systems)} systems, reconnected {len(reconnected_hosts)} hosts, validated {len(validated_scripting)} scripting environments"
            },
            'threat_intelligence': {
                'misp_enrichment': len([a for a in suspicious_activities if a.get('threat_intel')]),
                'high_confidence_indicators': len([a for a in suspicious_activities if a.get('threat_intel') and len(a['threat_intel']) > 0])
            }
        }
    )
    print(f"   Incident report created in IRIS: {iris_report.get('report_id', 'unknown')}")
except Exception as e:
    print(f"   Report creation failed: {e}")

# Share indicators with threat intelligence community
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
try:
    shared_iocs = []
    for activity in suspicious_activities:
        if activity.get('threat_intel') and len(activity['threat_intel']) > 0:
            # Share script hashes, command patterns, and associated IPs
            ioc_attributes = []
            if activity.get('script_hash'):
                ioc_attributes.append({
                    'type': 'sha256',
                    'value': activity['script_hash'],
                    'comment': f'Script hash from command execution incident {alert_data.get("incident_id", "unknown")}'
                })
            if activity.get('command_pattern'):
                ioc_attributes.append({
                    'type': 'text',
                    'value': activity['command_pattern'],
                    'comment': f'Command pattern from scripting interpreter abuse'
                })
            if activity.get('src_ip'):
                ioc_attributes.append({
                    'type': 'ip-src',
                    'value': activity['src_ip'],
                    'comment': f'Source IP associated with scripting execution activity'
                })

            if ioc_attributes:
                misp_event = misp.create_event(
                    title=f"Command and Scripting Interpreter Abuse: {activity.get('description', 'Unknown')}",
                    description=f"Command and scripting interpreter abuse detected during incident response. Activity: {activity.get('description', 'Unknown')}",
                    attributes=ioc_attributes
                )
                if misp_event.get('status') == 'created':
                    shared_iocs.extend([attr['value'] for attr in ioc_attributes])
                    print(f"   Shared {len(ioc_attributes)} IOCs for scripting execution activity")

    print(f"   Shared {len(shared_iocs)} indicators with threat intelligence community")
except Exception as e:
    print(f"   IOC sharing failed: {e}")

# Generate lessons learned
print(f"\n[ACTION] Generating lessons learned...")
lessons_learned = {
    'incident_type': 'Execution - Command and Scripting Interpreter',
    'key_findings': [
        f"Scripting execution indicators detected: {len(suspicious_activities)} across {len(affected_systems)} systems",
        f"Scripts executed: {sum(len(s.get('executed_scripts', [])) for s in affected_systems)} total",
        f"Most common interpreter: {max(set([a.get('interpreter', 'unknown') for a in suspicious_activities]), key=[a.get('interpreter', 'unknown') for a in suspicious_activities].count) if suspicious_activities else 'none'}",
        f"Threat intelligence enrichment: {len([a for a in suspicious_activities if a.get('threat_intel')])} activities enriched",
        f"Containment effectiveness: {len(isolated_hosts)}/{len(affected_systems)} systems isolated within detection window"
    ],
    'improvements_needed': [
        'Enhance script execution monitoring and logging',
        'Implement real-time command pattern analysis',
        'Strengthen interpreter security controls',
        'Improve automated script quarantine capabilities'
    ],
    'preventive_measures': [
        'Deploy advanced script execution protection',
        'Implement behavioral analytics for command anomalies',
        'Regular security policy updates for emerging scripting threats',
        'Enhanced threat intelligence integration for known malicious scripts'
    ]
}

try:
    iris.add_lessons_learned(
        incident_id=alert_data.get('incident_id', 'unknown'),
        lessons=lessons_learned
    )
    print(f"   Lessons learned documented in IRIS")
except Exception as e:
    print(f"   Lessons learned documentation failed: {e}")

# Final status update
print(f"\n Incident {alert_data.get('incident_id', 'unknown')} successfully resolved:")
print(f"  - Technique: Execution - Command and Scripting Interpreter (T1059)")
print(f"  - Status: Closed")
print(f"  - Systems Recovered: {len(restored_systems) + len(reconnected_hosts)}")
print(f"  - Scripts Neutralized: {len(removed_scripts)} malicious scripts removed")
print(f"  - Threat Indicators Shared: {len(shared_iocs) if 'shared_iocs' in locals() else 0}")
print(f"  - Incident Report Generated: Yes")
print(f"  - Lessons Learned Documented: Yes")

print(f"\n{'=' * 60}")
print("INCIDENT RESPONSE COMPLETE")
print(f"{'=' * 60}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
